# Earthcare

In [ ]:
from pystac_client import Client
from dotenv import load_dotenv
import earthcarekit as eck

import os
import requests
import tqdm

In [ ]:
bbox =[-30, -30, 30, 30]
datetime=['2025-09-21T00:00:00Z',
              '2025-09-21T23:59:59Z']
product = "AC__TC__2B"
var = "synergetic_target_classification"


load_dotenv()


CLIENT_ID = "offline-token"

CLIENT_SECRET = "p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"
MAAP_TOKEN = os.getenv("MAAP_TOKEN")
print(f"Using MAAP token: {MAAP_TOKEN[:10]}...")  # Print the first 10 characters for verification

In [ ]:

catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

# Select one or more collection(s)
EC_COLLECTION = ['EarthCAREL2Validated_MAAP']
EC_multiple_COLLECTIONS = [
    'EarthCAREL2Validated_MAAP', 'EarthCAREL1Validated_MAAP']

search = catalog.search(
    collections=EC_COLLECTION,
    # For example filter by product type and baseline. Use boolean logic for multi-filter queries
    filter="productType = 'AC__TC__2B' and productVersion = 'ba'",
    bbox=bbox,
    datetime=datetime,
    method='GET',  # This is necessary
    max_items=1000
    # max_items=5  # Adjust as needed, given the large amount of products it is recommended to set a limit if especially if you display results in pandas dataframe or similiar
)
items = list(search.items())  # Get all items as a list
# items = search.item_collection() # Get all items as a STAC ItemCollection
results = search.matched()

print(f"Number of items found: {results}")

In [ ]:
url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
data = {
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "grant_type": "refresh_token",
    "refresh_token": MAAP_TOKEN,
    "scope": "offline_access openid"
}

response = requests.post(url, data=data)
response.raise_for_status()

response_json = response.json()
access_token = response_json.get('access_token')
print("Access token retrieved successfully.")

if not access_token:
    raise RuntimeError("Failed to retrieve access token from IAM response")

In [ ]:
# 2. Download the H5 file from an item
for item in tqdm.tqdm(items):
    h5_url = item.assets["enclosure_h5"].href
    filename =os.path.join("./DATA/EC", item.assets["enclosure_h5"].extra_fields["file:local_path"])

    response = requests.get(
        h5_url,
        headers={"Authorization": f"Bearer {access_token}"},
        stream=True
    )
    with open(filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, ax = plt.subplots(
    subplot_kw={"projection": ccrs.PlateCarree()},
    figsize=(6, 5)
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0")
ax.add_feature(cfeature.OCEAN, facecolor="#d0e8f0")

for item in items:
    coords = item.geometry["coordinates"]  # [[lon, lat], ...]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    ax.plot(lons, lats, transform=ccrs.Geodetic(), color="crimson", linewidth=1.5)
    ax.plot(lons[0], lats[0], "o", color="crimson", ms=4, transform=ccrs.PlateCarree())

# Pad bbox for extent
lon0, lat0, lon1, lat1 = bbox
ax.set_extent([lon0 - 5, lon1 + 5, lat0 - 5, lat1 + 5], crs=ccrs.PlateCarree())

ax.set_title(f"EarthCARE ground tracks ({len(items)} passes)\n{datetime[0][:10]}", fontsize=10)
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:

with eck.read_product("./DATA/EC/ECA_EXBA_AC__TC__2B_20250921T002549Z_20250921T155724Z_07469H.h5") as ds:
    cfig = eck.CurtainFigure().ecplot(
        ds,
        var=var,
        height_range=(0, 20e3),  # Show only data within 20km height

    )
    cfig.ecplot_elevation(ds)  # NEW
    # cfig.ecplot_tropopause(ds)  # NEW

    # Save the image
    # eck.save_plot(cfig, filepath="curtain_with_elevation_and_tropopause.png")

# MTG

In [ ]:
import eumdac
import datetime
import shutil
import fnmatch
import requests
import time
import os
import zipfile
import json
import dotenv

In [ ]:
# Insert your personal key and secret into the single quotes
dotenv.load_dotenv()
consumer_key = os.getenv("EUMETSAT_KEY")
consumer_secret = os.getenv("EUMETSAT_SECRET")
print(f"Consumer key: {consumer_key}")
print(f"Consumer secret: {consumer_secret}")

credentials = (consumer_key, consumer_secret)

token = eumdac.AccessToken(credentials)

print(f"This token '{token}' expires {token.expiration}")

In [ ]:
datastore = eumdac.DataStore(token)

selected_collection = datastore.get_collection('EO:EUM:DAT:0662')
print(f"{selected_collection} - {selected_collection.title}")
print(f"Description: {selected_collection.abstract}")
print(f"Metadata: {selected_collection.metadata}")
print(f"Search options: {selected_collection.search_options} \n")

In [ ]:
import os
from shapely.wkt import loads
from shapely.geometry import Polygon

# Define file path to precomputed FCI data chunk map in WKT format
wkt_file_path = "FCI_chunks.wkt"

if not os.path.exists(wkt_file_path):
    raise FileNotFoundError(
        f"File {wkt_file_path} not found. Make sure it is in the repository.")

# Load WKT chunk footprints
with open(wkt_file_path, "r") as file:
    wkt_data = file.readlines()

# Parse chunk polygons from WKT
chunk_polygons = {}
for line in wkt_data:
    chunk_id, wkt_poly = line.strip().split(',', 1)  # Extract chunk ID
    chunk_polygons[chunk_id] = loads(wkt_poly)

print(f"Loaded {len(chunk_polygons)} chunk footprints from WKT file.")

In [ ]:
# Define ROI bounds (latitude and longitude bounding bbox)
user_roi = {
    "lat_min": 0,
    "lat_max": 15,
    "lon_min": 15,
    "lon_max": 30
}
print(f"Defined ROI: {user_roi}")

In [ ]:
# Convert user ROI to a Shapely Polygon
roi_polygon = Polygon([
    (user_roi["lon_min"], user_roi["lat_min"]),
    (user_roi["lon_min"], user_roi["lat_max"]),
    (user_roi["lon_max"], user_roi["lat_max"]),
    (user_roi["lon_max"], user_roi["lat_min"])
])

# Find chunks that intersect with ROI
relevant_chunks = []
for chunk_id, chunk_poly in chunk_polygons.items():
    if roi_polygon.intersects(chunk_poly):
        relevant_chunks.append(chunk_id)

print(
    f"Found {len(relevant_chunks)} chunks intersecting the ROI: {relevant_chunks}")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import LineString
from pyproj import CRS

wgs84_crs = CRS.from_proj4("+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs")

# Convert chunk polygons to a GeoDataFrame
gdf_chunks = gpd.GeoDataFrame({"chunk_id": list(chunk_polygons.keys(
)), "geometry": list(chunk_polygons.values())}, crs=wgs84_crs)
# Plot setup
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={
                       "projection": ccrs.PlateCarree()})
ax.set_extent([-90, 90, -90, 90])
ax.coastlines("50m", linewidth=0.25)
ax.add_feature(cfeature.LAND, facecolor="lightgray",
               edgecolor="black", linewidth=0.25)

# Plot chunks with labels
for i, row in gdf_chunks.iterrows():
    chunk_id, chunk_poly = row["chunk_id"], row["geometry"]
    if not chunk_poly.is_valid:
        continue
    ax.fill(*chunk_poly.exterior.xy,
            color=plt.cm.tab20.colors[i % 20], alpha=0.25, transform=ccrs.PlateCarree())

    # Label position inside polygon
    center_x = (chunk_poly.bounds[0] + chunk_poly.bounds[2]) / 2
    vertical_line = LineString(
        [(center_x, chunk_poly.bounds[1]), (center_x, chunk_poly.bounds[3])])
    label_y = vertical_line.intersection(chunk_poly).centroid.y
    ax.text(center_x, label_y, chunk_id, fontsize=6,
            transform=ccrs.PlateCarree(), ha="center", va="center")

# Highlight ROI
ax.plot(*roi_polygon.exterior.xy, color="red", linewidth=1,
        linestyle="dashed", transform=ccrs.PlateCarree())
plt.title("MTG Chunk coverage extent and user ROI")
plt.show()

In [ ]:
dtstart = datetime.datetime(2024, 11, 1, 11, 50)
dtend = datetime.datetime(2024, 11, 1, 12, 0)

print(f"Time window: from {dtstart} to {dtend}.")

In [ ]:
def download_chunks_in_time_window(selected_collection, dtstart, dtend, chunk_ids):
    """
    Search for products in the given time window, download relevant .nc entries and trailer chunk (0041).
    """

    # Always ensure trailer chunk "0041" is included
    chunk_ids.append("0041")

    chunk_patterns = [f"_{cid}.nc" for cid in chunk_ids]

    # Products in time window
    products = selected_collection.search(dtstart=dtstart, dtend=dtend)
    print(f"Found {len(products)} matching timestep(s).")

    # Filter relevant entries
    for product in products:
        for entry in product.entries:
            if any(pattern in entry for pattern in chunk_patterns):
                try:
                    with product.open(entry=entry) as fsrc:
                        local_filename = os.path.join("./DATA/MTG", os.path.basename(fsrc.name))
                        print(f"Downloading file {local_filename}...")
                        with open(local_filename, 'wb') as fdst:
                            shutil.copyfileobj(fsrc, fdst)
                        print(f"Saved file {local_filename}")
                except Exception as e:
                    print(f"Error downloading {entry}: {e}")


# Run the function
download_chunks_in_time_window(
    selected_collection=selected_collection, dtstart=dtstart, dtend=dtend, chunk_ids=relevant_chunks)

In [ ]:
import os
import numpy as np
import netCDF4 as nc4
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

MTG_DIR = "DATA/MTG"
CHANNEL = "ir_97"

# ── 1. Collect all BODY files present on disk ────────────────────────────────
body_files = sorted(
    [os.path.join(MTG_DIR, f) for f in os.listdir(MTG_DIR) if "BODY" in f]
)
print(f"Found {len(body_files)} BODY chunks")

# ── 2. Read and concatenate radiance + coordinates from each chunk ───────────
rad_chunks, y_chunks = [], []
x_rad = proj_params = None

for path in body_files:
    with nc4.Dataset(path) as ds:
        grp = ds[f"data/{CHANNEL}/measured"]

        rad = grp["effective_radiance"][:]
        y   = grp["y"][:]
        rad_chunks.append(rad)
        y_chunks.append(y)

        if x_rad is None:
            x_rad = grp["x"][:]

        if proj_params is None:
            p = ds["data"]["mtg_geos_projection"]
            proj_params = {k: getattr(p, k) for k in p.ncattrs()}

radiance = np.vstack(rad_chunks)
y_rad    = np.concatenate(y_chunks)

h    = proj_params["perspective_point_height"]
lon0 = proj_params["longitude_of_projection_origin"]

print(f"Radiance shape: {radiance.shape},  "
      f"range: {np.nanmin(radiance):.3f}–{np.nanmax(radiance):.3f} mW·m⁻²·sr⁻¹·(cm⁻¹)⁻¹")

# ── 3. Plot in native geostationary projection (avoids lon/lat NaN issues) ───
geos_crs = ccrs.Geostationary(
    central_longitude=lon0,
    satellite_height=h,
    sweep_axis="y",
)

# extent in metres on the projection plane
x_m = x_rad * h
y_m = y_rad * h
extent = [x_m.min(), x_m.max(), y_m.min(), y_m.max()]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": geos_crs})

vmin, vmax = np.nanpercentile(radiance, [5, 95])
im = ax.imshow(
    radiance,
    origin="upper",
    extent=extent,
    transform=geos_crs,
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
    interpolation="none",
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="red")
ax.add_feature(cfeature.BORDERS,   linewidth=0.3, edgecolor="red", linestyle=":")
ax.gridlines(draw_labels=True, linewidth=0.3, color="yellow", alpha=0.6)

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04,
             label="Radiance [mW·m⁻²·sr⁻¹·(cm⁻¹)⁻¹]")
ax.set_title(
    f"MTG/FCI  {CHANNEL.upper()}  |  {len(body_files)} BODY chunks  |  2024-11-01 ~11:50 UTC",
    fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
import netCDF4 as nc4
import numpy as np
import matplotlib.pyplot as plt

# ── Load a patch ─────────────────────────────────────────────────────────────
patch_path = "./DATA/patches/EC_MTG_patch_07166E_001_20250901T123158.nc"
ds = nc4.Dataset(patch_path)


# ── True Colour RGB (vis_06, vis_05, vis_04) ─────────────────────────────────
band_names = ["vis_06", "vis_05", "vis_04"]
bands = [ds[f"effective_radiance_{b}"][:].astype(np.float32) for b in band_names]

# Match resolutions: upsample IR/lower-res bands to the highest-res shape
max_shape = max(b.shape for b in bands)
for i, b in enumerate(bands):
    if b.shape != max_shape:
        zy, zx = max_shape[0] // b.shape[0], max_shape[1] // b.shape[1]
        bands[i] = np.repeat(np.repeat(b, zy, axis=0), zx, axis=1)

rgb = np.stack(bands, axis=-1)  # (H, W, 3)

# Per-band percentile stretch + gamma (same logic as viewer.py)
gamma = [1.0, 1.0, 1.0]
for i in range(3):
    band = rgb[:, :, i]
    valid = band[~np.isnan(band)]
    if len(valid) > 0:
        lo, hi = np.percentile(valid, [2, 98])
        rgb[:, :, i] = np.clip(band, lo, hi)
        rgb[:, :, i] = (rgb[:, :, i] - lo) / (hi - lo + 1e-10)
        if gamma[i] != 1.0:
            rgb[:, :, i] = np.power(rgb[:, :, i], 1.0 / gamma[i])
    else:
        rgb[:, :, i] = 0

rgb = np.nan_to_num(rgb, nan=0.0)

# ── Read geolocation for axis labels ─────────────────────────────────────────
lat = ds["latitude"][:]
lon = ds["longitude"][:]
center_lat = float(ds.getncattr("center_lat"))
center_lon = float(ds.getncattr("center_lon"))
ec_time = ds.getncattr("ec_time")
# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb, origin="upper", interpolation="none")
ax.set_title(f"True Colour (vis_06 / vis_05 / vis_04)\n"
             f"Center: ({center_lat:.2f}°, {center_lon:.2f}°)  |  EC time: {ec_time}",
             fontsize=11)
ax.set_xlabel("Pixel X")
ax.set_ylabel("Pixel Y")
plt.tight_layout()
plt.show()

In [ ]:
latitude = ds["latitude"][:]
longitude = ds["longitude"][:]

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0")
ax.add_feature(cfeature.OCEAN, facecolor="#d0e8f0")

# Plot the patch outline or points
ax.scatter(longitude, latitude, s=1, color="blue", transform=ccrs.PlateCarree(), alpha=0.5)
ax.plot(center_lon, center_lat, "ro", markersize=5, transform=ccrs.PlateCarree())

ax.set_extent([longitude.min() - 20, longitude.max() + 20, latitude.min() - 20, latitude.max() + 20], crs=ccrs.PlateCarree())
ax.set_title(f"Patch Location\nCenter: ({center_lat:.2f}°, {center_lon:.2f}°)", fontsize=12)
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
plt.tight_layout()
plt.show()